# Adding Tools & MCP 

Our simple **Authenticator Agent** reasoned over data we handed it directly in the prompt. In production, the agent will need to reach out to the Socotra policy system to verify identity. It is a **tool based** or **tool-augmented agent**, and this case, for MCP server. Other agents in the process need to complete specialized tasks around verification and document processing whichi also requires tools.

This notebook has three parts where you gradually build tool use capabilities for the agents:

- Part A: Add Python stub tools to the Authenticator. The agent will fetch policy data via tools instead of receiving it in the prompt.
- Part B: Swap the stubs for the live socotra_mock MCP server. One line changes. Everything else is identical.

### The MCP Mental Model
MCP (Model Context Protocol) is a standard that lets agents discover and call tools hosted on external servers — without knowing their implementation. Think of it as a USB-C standard for agent tools:

The MCP Server exposes tools with a schema the agent can read at runtime
The MCP Client (inside your Strands agent) handles the protocol
The agent decides when to call which tool based on its reasoning

#### Why Stubs First?
Before connecting to the real mock server, you'll build Python stub functions that match the MCP tool signatures. This lets you:

Understand exactly what data flows in and out of each tool
Test agent reasoning without a running server
See the exact moment the stub becomes a real MCP call — one line changes

### Setup
We will use Nova 2 Lite with reasoning enabled via additional_model_request_fields. We also set up the path to the socotra_mock MCP server, which lives in the repo alongside the notebooks.

In [ ]:
import boto3
import json
import sys
import os
from pathlib import Path
from strands import Agent, tool
from strands.models import BedrockModel

# ── Path setup ────────────────────────────────────────────────────────────────
# Notebooks:   insurance-claims-processing/notebooks/
# MCP servers: insurance-claims-processing/mcp_servers/socotra_mock/
REPO_ROOT       = Path("../..").resolve()
MCP_SERVER_PATH = REPO_ROOT / "mcp_servers" / "socotra_mock"
MOCK_DATA_PATH  = MCP_SERVER_PATH / "mock_data.json"

sys.path.insert(0, str(MCP_SERVER_PATH))


#Nova 2 Lite ID
MODEL_ID = "us.amazon.nova-2-lite-v1:0"  
REGION   = "us-east-1"

#create a Strands BedrockModel provider instance
model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature = 0.1,
    top_p=0.9,
    additional_request_fields={
        "reasoningConfig": {
            "type": "enabled",
            "maxReasoningEffort": "low" # note Temperature, topP and topK cannot be used if maxReasoningEffort is set to high
        }
    }
)

print(f"✅ Nova 2 Lite configured with reasoning enabled")
print(f"   Model           : {MODEL_ID}")
print(f"   MCP server path : {MCP_SERVER_PATH}")


#### Load Mock Policy Data

First, you will use the Strands @tool decorator to define stub tools for the **Authenticator Agent**. You will test how the agent functions with these tools before working with MCP. The socotra_mock MCP server loads mock_data.json at startup. We load the same file here so our stub tools return identical data to what the 'real' MCP server returns, making the stubs faithful to the real tool contract 

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load the same mock_data.json used by the real Socotra mock MCP server.
# This ensures stubs return identical data to what the MCP server returns.
# Key discipline: stubs must be faithful to the real tool contract.
# ─────────────────────────────────────────────────────────────────────────────

with open(MOCK_DATA_PATH) as f:
    raw = json.load(f)

# mock_data.json is a list with one object containing a "policies" array
MOCK_DB = {p["policy_number"]: p for p in raw.get("policies", [])}

print("✅ Mock database loaded from socotra_mock/mock_data.json")
print(f"   Policies on file: {len(MOCK_DB)}")
print()
for pnum, policy in MOCK_DB.items():
    print(f"   {pnum} | {policy['insured_owner']['name']} | status={policy['status']}")


## Part A — Stub Tools: Upgrading the Authenticator
The previous Authenticator agent received the policy record directly in the request. Now we give it four tools that mirror the Socotra MCP server's tool signatures exactly. The agent will call these tools to fetch what it needs.

We use the Strands @tool decorator to register each function with Strands so the agent can discover and invoke it. Notice that the function signatures and return shapes below are identical to the tools defined in socotra_mock/server.py — that's intentional.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# These stubs mirror the tools defined in socotra_mock/server.py:
#   verify_beneficiary_identity  → identity match + contact details
#   verify_coverage_status       → active / lapsed check
#   check_exclusions             → death circumstances vs policy exclusions
#   verify_beneficiary_details   → full beneficiary contact info
#
# The @tool decorator registers them with Strands so the agent can call them.
# Return shapes are IDENTICAL to what the MCP server returns.
# ─────────────────────────────────────────────────────────────────────────────

@tool
def verify_beneficiary_identity(policy_number: str, claimant_name: str) -> dict:
    """
    Verify beneficiary identity against policy records.

    Args:
        policy_number: Policy number (e.g., POL-WL-2024-001)
        claimant_name: Name of person filing the claim

    Returns:
        Verification result with identity_verified, policy_match, beneficiary details
    """
    print(f"   [STUB] verify_beneficiary_identity(policy={policy_number}, claimant={claimant_name})")

    policy = MOCK_DB.get(policy_number)
    if not policy:
        return {
            "identity_verified": False,
            "policy_match": False,
            "error": f"Policy {policy_number} not found",
            "confidence": 0.0
        }

    beneficiaries = policy.get("beneficiaries", [])
    matched = None
    for b in beneficiaries:
        if claimant_name.lower() in b["name"].lower() or \
           b["name"].lower() in claimant_name.lower():
            matched = b
            break

    if matched:
        return {
            "identity_verified": True,
            "policy_match": True,
            "beneficiary_name": matched["name"],
            "relationship": matched["relationship"],
            "percentage": matched["percentage"],
            "contact": matched["contact"],
            "confidence": 0.95,
            "verification_method": "socotra_stub"
        }
    return {
        "identity_verified": False,
        "policy_match": False,
        "error": f"Claimant '{claimant_name}' not found in beneficiaries",
        "beneficiaries_on_file": [b["name"] for b in beneficiaries],
        "confidence": 0.2,
        "verification_method": "socotra_stub"
    }


@tool
def verify_coverage_status(policy_number: str) -> dict:
    """
    Check if a policy is active or lapsed.

    Args:
        policy_number: Policy number

    Returns:
        Coverage status with active flag, status, and relevant dates
    """
    print(f"   [STUB] verify_coverage_status(policy={policy_number})")

    policy = MOCK_DB.get(policy_number)
    if not policy:
        return {"coverage_active": False, "status": "not_found",
                "error": f"Policy {policy_number} not found"}

    status    = policy.get("status", "unknown")
    is_active = status == "active"

    result = {
        "coverage_active": is_active,
        "status": status,
        "policy_type": policy.get("policy_type"),
        "issue_date": policy.get("issue_date"),
        "premium_status": policy.get("premium_status")
    }
    if status == "lapsed":
        result["lapse_date"] = policy.get("lapse_date")
        result["reason"] = "Premium payments not current"
    return result


@tool
def check_exclusions(policy_number: str, death_circumstances: str) -> dict:
    """
    Check if death circumstances trigger any policy exclusions.

    Args:
        policy_number: Policy number
        death_circumstances: Description of death circumstances from the claim

    Returns:
        Exclusion check result with triggered exclusions and claim eligibility
    """
    print(f"   [STUB] check_exclusions(policy={policy_number})")

    policy = MOCK_DB.get(policy_number)
    if not policy:
        return {"exclusions_triggered": [], "claim_eligible": True,
                "error": f"Policy {policy_number} not found"}

    exclusions = policy.get("exclusions", [])
    triggered  = []
    circ       = death_circumstances.lower()

    if "suicide" in circ and "suicide_within_2_years" in exclusions:
        triggered.append({
            "exclusion": "suicide_within_2_years",
            "description": "Suicide within 2 years of policy issue",
            "requires_review": True
        })
    if ("war" in circ or "terrorism" in circ) and "war_or_terrorism" in exclusions:
        triggered.append({
            "exclusion": "war_or_terrorism",
            "description": "Death due to war or terrorism",
            "requires_review": True
        })
    if "aviation" in circ and "aviation_non_commercial" in exclusions:
        triggered.append({
            "exclusion": "aviation_non_commercial",
            "description": "Non-commercial aviation accident",
            "requires_review": True
        })

    return {
        "exclusions_triggered": triggered,
        "claim_eligible": len(triggered) == 0,
        "all_exclusions": exclusions,
        "requires_manual_review": len(triggered) > 0
    }


@tool
def verify_beneficiary_details(policy_number: str) -> dict:
    """
    Retrieve complete beneficiary information including contact details.

    Args:
        policy_number: Policy number

    Returns:
        List of beneficiaries with contact details and allocation percentages
    """
    print(f"   [STUB] verify_beneficiary_details(policy={policy_number})")

    policy = MOCK_DB.get(policy_number)
    if not policy:
        return {"beneficiaries": [], "error": f"Policy {policy_number} not found"}

    beneficiaries = policy.get("beneficiaries", [])
    return {
        "policy_number": policy_number,
        "beneficiaries": beneficiaries,
        "total_percentage": sum(b.get("percentage", 0) for b in beneficiaries)
    }


print("✅ Stub tools defined — matching socotra_mock/server.py signatures exactly:")
print("   verify_beneficiary_identity  → identity match + contact details")
print("   verify_coverage_status       → active / lapsed check")
print("   check_exclusions             → death circumstances vs policy exclusions")
print("   verify_beneficiary_details   → full beneficiary contact info")


**Upgrade the Authenticator's system prompt:**
This prompt version tells the agent to use its tools to fetch what it needs, instead of reasoning over data passed in the requst as done with the simple version of the agent. That's the architectural shift from a reasoning agent to a tool-augmented agent. Also note, the instruction in the prompt that informs the agents on how it should use it's tools

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Key change:
#   Simple Authenticator system prompt → "reason over data I give you in the request"
#   Tool based authenticator system prompt → "use your tools to fetch what you need"
#
# The validation logic is identical. Only the data sourcing changes.
# ─────────────────────────────────────────────────────────────────────────────

AUTHENTICATOR_SYSTEM_PROMPT_V2 = """You are the Authenticator Agent for Insurance Claims Processing.

Your role is to validate beneficiary identity against the Socotra policy system
and confirm claim submission completeness before the claim proceeds to document processing.

## Your Validation Workflow

Use your tools in this sequence for every claim:

1. **verify_coverage_status(policy_number)**
   - Confirm the policy is active before doing anything else
   - If lapsed or not found: set status FAILED, recommend rejection

2. **verify_beneficiary_identity(policy_number, claimant_name)**
   - Confirm the claimant matches the beneficiary on record
   - Capture confidence score — flag anything below 0.85 for human review

3. **check_exclusions(policy_number, death_circumstances)**
   - Check whether the stated cause/circumstances of death trigger any exclusions
   - Any triggered exclusion requires escalation, not auto-rejection

4. **verify_beneficiary_details(policy_number)**
   - Retrieve full contact details for the verified beneficiary
   - These are needed by the Communicator Agent in Phase 3

5. **Completeness Check (no tool needed)**
   - Compare submitted documents against the required documents list
   - Identify missing required documents

## Output Format

AUTHENTICATION STATUS: VERIFIED | PARTIAL | FAILED
COVERAGE CHECK: [Active/Lapsed/Not Found] — [policy_type, issue_date]
IDENTITY CHECK: [Verified/Failed] — [confidence score, relationship]
EXCLUSIONS CHECK: [Clear/Triggered] — [list triggered exclusions if any]
COMPLETENESS CHECK: [Complete/Incomplete] — [missing documents if any]
BENEFICIARY CONTACT: [name, email, phone, address]
RECOMMENDED ACTION: Proceed to Document Processing | Request Additional Info | Escalate to Adjudicator | Reject
NOTES: [flags, low-confidence fields, anomalies for the adjudicator]

## Guidelines
- Always run all four tool calls — do not short-circuit on a single failure
- You validate identity and coverage — you do NOT make final coverage decisions
- Low confidence identity matches (<0.85) must be flagged, not silently accepted
- Protect PII in your output — do not repeat SSNs or full DOBs unnecessarily
"""

print(f"✅ Upgraded system prompt defined ({len(AUTHENTICATOR_SYSTEM_PROMPT_V2)} characters)")
print("   Key change: agent now uses tools to fetch policy data")
print("   rather than reasoning over data passed directly in the request")


**Build the Tool-Augmented Authenticator:** Build the upgraded Authenticator with stub tools. Note the added tools=[...] list, Strands registers each tool's schema so the agent can discover and call them during its reasoning loop.

In [ ]:
authenticator_v2 = Agent(
    model=model,
    system_prompt=AUTHENTICATOR_SYSTEM_PROMPT_V2,
    tools=[
        verify_beneficiary_identity,
        verify_coverage_status,
        check_exclusions,
        verify_beneficiary_details,
    ]
)

print("✅ Tool-augmented Authenticator Agent built")
print()

# ── New claim ──────────────────────────────────────────────────────────
# Notice: the request no longer includes policy record data for comparison.
# The agent fetches everything from the database using its tools.

NEW_CLAIM_SUBMISSION = """
## Authentication Request

**Claim ID:** CLM-2025-00847
**Policy Number:** POL-WL-2024-001
**Claimant Name:** Sarah Jane Smith
**Date of Death:** 2025-01-10
**Cause / Circumstances of Death:** Natural causes — cardiac arrest at home

### Documents Submitted
- death_certificate
- claim_form
- beneficiary_id

### Required Documents
- death_certificate
- claim_form
- beneficiary_id
- policy_document

Please authenticate this claim.
"""

print("🤖 Running Authenticator Agent...")
print("=" * 60)
print("💡 Watch the output — each [STUB] line is a tool invocation by the agent")
print()

response = authenticator_v2(NEW_CLAIM_SUBMISSION)

print()
print("=" * 60)
print("✅ Authentication complete")

**Inspect the Reasoning + Tool Call Trace**
The trace shows you three things happening in sequence:

- Reasoning (reasoningContent block) — Nova 2 Lite thinks through which tools to call and in what order before making any calls
- Tool calls (tool_use blocks) — the agent invokes each tool with specific inputs it derived from the claim
- Final assessment (text block) — the structured verdict, written after all tool results are in  
The agent sequenced all four tool calls based on its system prompt instruction

In [ ]:
#note reasoning trace is truncated for brevity

print("🔍 Full Agent Trace")
print()

parsed_anything = False

for i, message in enumerate(authenticator_v2.messages):
    role    = message.get("role", "unknown")
    content = message.get("content", [])

    # Normalize content to a list of blocks
    if isinstance(content, str):
        blocks = [{"type": "text", "text": content}]
    elif isinstance(content, list):
        blocks = content
    else:
        blocks = [{"type": "text", "text": str(content)}]

    print(f"── Message {i} | role={role} ──────────────────────────────")

    for block in blocks:
        if not isinstance(block, dict):
            print(f"   [raw block] {str(block)[:200]}")
            parsed_anything = True
            continue

        block_type = block.get("type", "unknown")

        # ── Nova 2 reasoning block — show in full ─────────────────────────
        if block_type == "reasoningContent":
            parsed_anything = True
            reasoning_raw = block.get("reasoningText", "")
            if isinstance(reasoning_raw, dict):
                reasoning_text = reasoning_raw.get("text", str(reasoning_raw))
            else:
                reasoning_text = str(reasoning_raw)
            print(f"🧠 REASONING ({len(reasoning_text)} chars):")
            # ── Option 3: full reasoning, no truncation ───────────────────
            print(reasoning_text)
            print()

        # ── Tool call (agent → tool) ──────────────────────────────────────
        elif block_type == "tool_use":
            parsed_anything = True
            print(f"🔧 TOOL CALL : {block.get('name', 'unknown')}")
            print(f"   Tool ID   : {block.get('id', 'n/a')}")
            print(f"   Input     : {json.dumps(block.get('input', {}), indent=2)}")
            print()

        # ── Tool result (tool → agent) — truncated at 300 chars ──────────
        elif block_type == "tool_result":
            parsed_anything = True
            result_content = block.get("content", "")
            if isinstance(result_content, list):
                for rc in result_content:
                    if isinstance(rc, dict) and rc.get("type") == "text":
                        result_preview = rc.get("text", "")[:300]
                    else:
                        result_preview = str(rc)[:300]
            else:
                result_preview = str(result_content)[:300]
            full_len = len(str(result_content))
            print(f"   ↳ Tool result: {result_preview}{'...' if full_len > 300 else ''}")
            print()

        # ── Final text response ───────────────────────────────────────────
        elif block_type == "text":
            parsed_anything = True
            text = block.get("text", "")
            if text.strip():
                print(f"📋 TEXT ({role}):")
                print(text)
                print()

        # ── Unknown block type — print raw ────────────────────────────────
        else:
            parsed_anything = True
            print(f"   [block type={block_type}] {str(block)[:200]}")
            print()

print()

# ── Fallback: raw dump if nothing parsed ─────────────────────────────────────
if not parsed_anything:
    print("⚠️  No structured blocks found. Raw messages dump:")
    print()
    for i, msg in enumerate(authenticator_v2.messages):
        print(f"Message {i}: {json.dumps(msg, default=str, indent=2)[:500]}")
        print()


**Edge Case: Lapsed Policy:** Now test with a lapsed policy. 

In [ ]:
LAPSED_POLICY_CLAIM = """
## Authentication Request

**Claim ID:** CLM-2025-00901
**Policy Number:** POL-WL-2023-042
**Claimant Name:** Emily Wilson
**Date of Death:** 2025-02-01
**Cause / Circumstances of Death:** Natural causes

### Documents Submitted
- death_certificate
- claim_form

### Required Documents
- death_certificate
- claim_form
- beneficiary_id
- policy_document

Please authenticate this claim.
"""

print("🔴 Testing lapsed policy scenario...")
print()
response_lapsed = authenticator_v2(LAPSED_POLICY_CLAIM)




## Part B — Replace Stubs with the Socotra Mock MCP Server
Note that the agent code, system prompt, and claim requests are identical to Part A. The only thing that changes is tools=[...] — from stub functions to tools discovered from the live MCP server at runtime.

In [ ]:
import sys
import os
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

SOCOTRA_SERVER_SCRIPT = str(MCP_SERVER_PATH / "server.py")

socotra_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command=sys.executable,         
            args=[SOCOTRA_SERVER_SCRIPT],
            env={
                "MOCK_DATA_PATH": str(MOCK_DATA_PATH),
                **os.environ
            }
        )
    )
)

print("✅ MCP client configured")
print(f"   Python : {sys.executable}")
print(f"   Server : {SOCOTRA_SERVER_SCRIPT}")





**Verify the Server Starts Cleanly:**
Before connecting via the MCP client, run this diagnostic to confirm the server starts without errors. A healthy server will print its startup banner to stderr and then block waiting for JSON-RPC input — the timeout=5 will kill it after 5 seconds, which is expected. You're looking for no errors in the stderr output and a non-zero return code (meaning it was still running when we killed it).

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, SOCOTRA_SERVER_SCRIPT],
    capture_output=True,
    text=True,
    timeout=5,
    cwd=str(MCP_SERVER_PATH)
)

print("=== STDERR (startup output) ===")
print(result.stderr[:1000] if result.stderr else "(empty — server may have crashed)")
print()
print(f"Return code: {result.returncode}")

**Run the Authenticator with mock Soctora server on MCP:**  
Now connect to the live server and run the same claim submission from Part A. Watch the output — the tool calls will look identical to the stub version, but they're now being served by the running MCP server process.

In [ ]:
with socotra_mcp_client:
    # Discover available tools from the running MCP server
    mcp_tools = socotra_mcp_client.list_tools_sync()

    print("🔧 Tools discovered from Socotra MCP server:")
    for t in mcp_tools:
        print(f"   - {t.tool_name}")
    print()

    # Build the MCP-powered Authenticator
    # System prompt is IDENTICAL to Part A
    authenticator_mcp = Agent(
        model=model,
        system_prompt=AUTHENTICATOR_SYSTEM_PROMPT_V2,
        tools=mcp_tools,   # ← Only change from Part A
    )

    print("🤖 Running Authenticator Agent (with MCP)")
    print("=" * 60)

    response_mcp = authenticator_mcp(NEW_CLAIM_SUBMISSION)

    print()
    print("=" * 60)
    print("✅ MCP-powered authentication complete")

print()
print("💡 What just changed:")
print("   BEFORE: tools=[verify_beneficiary_identity, verify_coverage_status, ...]")
print("   AFTER:  tools=mcp_tools  (discovered from MCP server at runtime)")
print("   The agent, system prompt, and claim request are identical.")


## Notebook Complete

You have integrated a tool on MCP server for the **Authenticator Agent**.
                                        
### What You Built
- An `Agent` that queries a live policy system hosted on MCP server
- Authentication of a claimant using data and tools on MCP a

### What This Agent Cannot Do Yet

These gaps are intentional — each one motivates a future notebook:

| Limitation | Why It Matters | 
|---|---|
| **No document inspection** | The agent trusts the claimant's word on submitted docs; it can't read them |
| **No PII protection** | DOB and names flow in plaintext — a HIPAA violation in production |
| **No session persistence** | Agent state is lost when the kernel restarts|
| **No structured handoff** | The agent returns text — nothing downstream acts on it automatically |
